# Semestrální projekt (Část 1): Exploratory Data Analysis & Clustering
**Student:** Diana Kasymbekova (KAS0301)
**Dataset:** Melbourne Housing Snapshot

---

## 1. Úvod a cíle projektu

V této části projektu se zaměříme na analýzu dat (EDA) a shlukování (clustering) nemovitostí v Melbourne.

Naším cílem je prozkoumat data a splnit následující úkoly:

1. **Data Cleaning & Preprocessing:** Vyčištění dat, zpracování chybějících a neobvyklých hodnot.
2. **Feature Engineering:** Vytvoření 2–3 nových proměnných podle vlastností trhu s nemovitostmi.
3. **Deep EDA:** Analýza rozdělení dat, vztahů mezi proměnnými a korelací. Každý graf doplníme vysvětlením.
4. **Cluster Analysis:** Rozdělení nemovitostí do skupin pomocí shlukovacích algoritmů a matematické zdůvodnění počtu skupin.

In [35]:
# Import základních knihoven pro práci s daty
import pandas as pd
import numpy as np

# Import knihoven pro vizualizaci
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Načtení a základní kontrola dat (Dataset Validation)
Podle požadavků projektu musí vybraný dataset splňovat přísná kritéria:
* Minimálně 2000 řádků a 10 příznaků.
* Přítomnost spojitých (continuous) a kategorických proměnných.
* Data musí být "surová" (přítomnost chybějících hodnot a nezpracovaných formátů).

Načteme data a programově potvrdíme, že dataset splňuje všechny podmínky.

In [36]:
# Načtení datasetu (soubor musí být ve stejné složce jako notebook)
df = pd.read_csv("melb_data.csv")

# Kontrola velikosti tabulky
print("=== VELIKOST DATASETU ===")
print(f"Počet pozorování (řádků): {df.shape[0]}")
print(f"Počet příznaků (sloupců): {df.shape[1]}\n")

# Zobrazení prvních 3 řádků pro vizuální seznámení se strukturou
display(df.head(3))

=== VELIKOST DATASETU ===
Počet pozorování (řádků): 13580
Počet příznaků (sloupců): 21



,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0


### 2.2. Analýza datových typů a detekce "špinavých" dat
Abychom mohli naplánovat strategie čištění (imputation strategies), musíme zjistit, ve kterých příznacích chybí hodnoty a jaké datové typy (např. data prodeje) jsou uloženy jako běžný text.

In [37]:
# Kontrola datových typů ve sloupcích
print("=== DATOVÉ TYPY A POČET VYPLNĚNÝCH HODNOT ===")
df.info()

print("\n" + "="*50 + "\n")

# Výpočet prázdných hodnot (zobrazíme pouze sloupce, kde reálně chybí data)
print("=== POČET CHYBĚJÍCÍCH HODNOT (NaN) VE SLOUPCÍCH ===")
missing_values = df.isna().sum().sort_values(ascending=False)
missing_only = missing_values[missing_values > 0]

# Vytvoření přehledné tabulky s procentuálním podílem chybějících dat
missing_df = pd.DataFrame({
    'Chybějící hodnoty': missing_only,
    'Procento z celku (%)': (missing_only / len(df) * 100).round(2)
})

display(missing_df)

=== DATOVÉ TYPY A POČET VYPLNĚNÝCH HODNOT ===
<class 'pandas.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         13580 non-null  str    
 1   Address        13580 non-null  str    
 2   Rooms          13580 non-null  int64  
 3   Type           13580 non-null  str    
 4   Price          13580 non-null  float64
 5   Method         13580 non-null  str    
 6   SellerG        13580 non-null  str    
 7   Date           13580 non-null  str    
 8   Distance       13580 non-null  float64
 9   Postcode       13580 non-null  float64
 10  Bedroom2       13580 non-null  float64
 11  Bathroom       13580 non-null  float64
 12  Car            13518 non-null  float64
 13  Landsize       13580 non-null  float64
 14  BuildingArea   7130 non-null   float64
 15  YearBuilt      8205 non-null   float64
 16  CouncilArea    12211 non-null  str    
 17  Lattitude      

,Chybějící hodnoty,Procento z celku (%)
BuildingArea,6450,47.50
YearBuilt,5375,39.58
CouncilArea,1369,10.08
Car,62,0.46


## 2.3. Kontrola duplicitních záznamů
Ověříme, zda dataset neobsahuje identické řádky, které by mohly zkreslit statistiky.

In [38]:
# Kontrola duplicit v datech
pocet_duplicit = df.duplicated().sum()
print(f"Počet plně duplicitních řádků: {pocet_duplicit}")

# Pokud by tam byly, odstraníme je:
if pocet_duplicit > 0:
    df = df.drop_duplicates()
    print("Duplicity byly odstraněny.")

Počet plně duplicitních řádků: 0


### 3. Zpracování chybějících hodnot (Data Cleaning & Imputation)

**3.1. Zpracování sloupce `BuildingArea` (obytná plocha)**

Ve sloupci `BuildingArea` chybí **6 450 hodnot (47,5 %)**. Jednoduché nahrazení všech těchto hodnot jedním průměrem by mohlo výrazně zmenšit přirozený rozptyl dat. Sloupec ale nemůžeme jednoduše odstranit, protože obytná plocha je důležitá pro analýzu trhu s nemovitostmi.

**Zvolená strategie:** Chybějící plochu nejprve doplníme mediánem pro skupinu se stejným typem nemovitosti (`Type`) a stejným počtem pokojů (`Rooms`). Pokud pro některou skupinu medián není k dispozici, použijeme medián podle typu nemovitosti a nakonec celkový medián sloupce. Tím omezíme počet ztracených pozorování a zachováme logiku podobných nemovitostí.

In [ ]:
# 1. Zkontrolujeme původní stav chybějících hodnot
print("Původní počet chybějících hodnot v BuildingArea:", df['BuildingArea'].isna().sum())

# 2. Imputace mediánem podle typu nemovitosti a počtu pokojů
df['BuildingArea'] = df['BuildingArea'].fillna(
    df.groupby(['Type', 'Rooms'])['BuildingArea'].transform('median')
)

# 3. Pokud některá skupina nemá medián, použijeme medián podle typu nemovitosti
df['BuildingArea'] = df['BuildingArea'].fillna(
    df.groupby('Type')['BuildingArea'].transform('median')
)

# 4. Poslední záložní možnost: celkový medián obytné plochy
df['BuildingArea'] = df['BuildingArea'].fillna(df['BuildingArea'].median())

print("Zbývající počet chybějících hodnot v BuildingArea:", df['BuildingArea'].isna().sum())

### 3.2. Zpracování sloupce `YearBuilt` (rok výstavby)

Ve sloupci `YearBuilt` chybí více než 5 300 hodnot. Jediný celkový průměr by mohl zkreslit rozdíly mezi lokalitami a typy nemovitostí.

**Zvolená strategie:** Chybějící hodnoty nejprve nahradíme mediánem pro kombinaci čtvrti (`Suburb`) a typu nemovitosti (`Type`). Pokud tato skupina neobsahuje žádnou známou hodnotu, použijeme medián podle typu a nakonec celkový medián sloupce.

### 3.3. Formátování data a další chybějící hodnoty

Sloupec `Date` převedeme z textu na typ `datetime`. Chybějící hodnoty v `CouncilArea` nebudeme důvodně mazat, protože tento sloupec není vstupem do clusterování; označíme je jako `Unknown`. Malý počet chybějících hodnot ve sloupci `Car` doplníme mediánem podle typu nemovitosti a případně celkovým mediánem.

Nakonec odstraníme pouze řádky, kterým chybí hodnoty ve sloupcích nezbytných pro plánovanou analýzu. Tím se vyhneme zbytečné ztrátě přibližně 10 % pozorování jen kvůli neznámému `CouncilArea`. Počet zachovaných řádků ověříme ve výstupu následující buňky.

In [ ]:
# 1. Imputace YearBuilt mediánem podle čtvrti a typu nemovitosti
df['YearBuilt'] = df['YearBuilt'].fillna(
    df.groupby(['Suburb', 'Type'])['YearBuilt'].transform('median')
)

# 2. Záložní imputace podle typu nemovitosti
df['YearBuilt'] = df['YearBuilt'].fillna(
    df.groupby('Type')['YearBuilt'].transform('median')
)

# 3. Poslední záložní možnost: celkový medián
df['YearBuilt'] = df['YearBuilt'].fillna(df['YearBuilt'].median())

# 4. Převod data prodeje z textu na datetime
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')

# 5. Neznámou oblast zachováme jako kategorii místo mazání řádků
df['CouncilArea'] = df['CouncilArea'].fillna('Unknown')

# 6. Malý počet chybějících hodnot v Car doplníme mediánem podle typu
#    a případně celkovým mediánem
df['Car'] = df['Car'].fillna(
    df.groupby('Type')['Car'].transform('median')
)
df['Car'] = df['Car'].fillna(df['Car'].median())

# 7. Odstraníme pouze řádky s chybějícími hodnotami v potřebných sloupcích
required_columns = [
    'Price', 'Rooms', 'Type', 'Date', 'YearBuilt', 'BuildingArea',
    'Bathroom', 'Distance', 'Landsize', 'Regionname'
]
df_clean = df.dropna(subset=required_columns).copy().reset_index(drop=True)

print("=== KONTROLA DAT PO ČIŠTĚNÍ ===")
print(f"Počet původních řádků: {len(df)}")
print(f"Počet řádků zachovaných pro analýzu: {len(df_clean)}")
print(f"Počet řádků odstraněných kvůli chybějícím hodnotám v nezbytných sloupcích: {len(df) - len(df_clean)}")
print("Chybějící hodnoty v klíčových sloupcích:")
display(df_clean[required_columns].isna().sum().to_frame('Počet NaN'))

## 4. Tvorba nových příznaků (Feature Engineering)

Podle zadání vytvoříme alespoň 2–3 nové proměnné založené na logice trhu s nemovitostmi:

1. **`BuildingAge` (stáří budovy):** Počet let mezi rokem prodeje a rokem výstavby. Věk budovy je pro interpretaci srozumitelnější než samotný rok výstavby.
2. **`PricePerArea` (cena za m²):** Cena nemovitosti dělená obytnou plochou. Umožňuje porovnat cenu objektů různých velikostí. Tento příznak použijeme při EDA, ale vynecháme jej z clusterování, abychom skupiny netvořili přímo podle ceny.
3. **`BathroomPerRoom` (počet koupelen na pokoj):** Poměrový ukazatel, který může přibližně vystihovat komfort dispozice. Protože je odvozen z `Bathroom` a `Rooms`, při použití v clusterování částečně zvýrazní tyto informace znovu; jde o omezení, které je třeba při interpretaci zohlednit.

In [ ]:
# 1. Stáří budovy v době prodeje
df_clean['BuildingAge'] = df_clean['Date'].dt.year - df_clean['YearBuilt']

# 2. Cena za metr čtvereční; nulovou plochu nepoužijeme jako jmenovatel
df_clean['PricePerArea'] = df_clean['Price'] / df_clean['BuildingArea'].replace(0, np.nan)

# 3. Poměr koupelen k počtu pokojů; ošetříme případ Rooms = 0
df_clean['BathroomPerRoom'] = (
    df_clean['Bathroom'] / df_clean['Rooms'].replace(0, np.nan)
)

# Odstraníme nekonečné nebo chybějící hodnoty pouze v nově vytvořených příznacích
df_clean = df_clean.replace([np.inf, -np.inf], np.nan)
df_clean = df_clean.dropna(
    subset=['BuildingAge', 'PricePerArea', 'BathroomPerRoom']
).copy().reset_index(drop=True)

print("=== KONTROLA NOVÝCH PŘÍZNAKŮ ===")
display(df_clean[['BuildingAge', 'PricePerArea', 'BathroomPerRoom']].head())
print(f"Počet pozorování po vytvoření příznaků: {len(df_clean)}")

### 4.1. Odstranění extrémních a nepravděpodobných hodnot (Outlier Handling)

V datech se mohou nacházet hodnoty, které jsou pro běžný rezidenční trh nepravděpodobné nebo mohou představovat chybu, například velmi starý rok výstavby či extrémně malá obytná plocha.

Použijeme předem stanovené doménové filtry:
- rok výstavby musí být alespoň 1850;
- stáří budovy nesmí být záporné;
- obytná plocha musí být v rozmezí 15–1 500 m²;
- velikost pozemku musí být v rozmezí 0–10 000 m²;
- cena a vzdálenost od centra musí být nezáporné a cena musí být kladná.

Tyto hranice jsou zvolené pro analýzu běžného rezidenčního trhu. Mohou vyloučit i některé skutečné, ale neobvyklé nemovitosti, takže výsledky nebudou popisovat úplně všechny možné typy objektů.

In [ ]:
# Doménové filtry pro hodnoty, které jsou neplatné nebo mimo zvolený rozsah
valid_mask = (
    (df_clean['Price'] > 0) &
    (df_clean['Distance'] >= 0) &
    (df_clean['YearBuilt'] >= 1850) &
    (df_clean['BuildingAge'] >= 0) &
    (df_clean['BuildingArea'] >= 15) &
    (df_clean['BuildingArea'] <= 1500) &
    (df_clean['Landsize'] >= 0) &
    (df_clean['Landsize'] <= 10000)
)
df_clean = df_clean.loc[valid_mask].copy().reset_index(drop=True)

print(f"Konečný počet vyčištěných pozorování: {len(df_clean)}")
print("Deskriptivní statistiky po aplikaci filtrů:")
display(df_clean[['Price', 'BuildingArea', 'Landsize', 'BuildingAge']].describe().round(2))

*Nové příznaky jsou vytvořeny a data prošla imputací i doménovými filtry. V následujících částech budeme zkoumat jejich rozdělení a vztahy; všechny interpretace je potřeba číst společně s grafy a uvedenými omezeními čištění.*

## 5. Průzkumná analýza dat (Deep Exploratory Data Analysis)

V této části zkoumáme rozdělení důležitých proměnných a jejich vzájemné vztahy. Cílem je najít vzorce, které pomohou porozumět trhu s nemovitostmi v Melbourne a připravit vhodné vstupy pro shlukování.

### 5.1. Univariační analýza: Rozdělení cen nemovitostí (`Price`)

Cena je jedním z hlavních ukazatelů trhu. Sledujeme tvar rozdělení, typickou cenovou úroveň a případný dlouhý chvost drahých nemovitostí.

In [ ]:
# Histogram cen nemovitostí s odhadem hustoty
plt.figure(figsize=(12, 6))
sns.histplot(df_clean['Price'], bins=60, kde=True, color='teal')
plt.title('Rozdělení cen nemovitostí v Melbourne', fontsize=16)
plt.xlabel('Cena (osa je zobrazena v milionech AUD)', fontsize=12)
plt.ylabel('Počet nemovitostí', fontsize=12)

# Zobrazení hodnot osy X v milionech AUD a odstranění záporného rozsahu osy
xticks = plt.xticks()[0]
plt.xticks(ticks=xticks, labels=[f'{x/1e6:.1f}M' for x in xticks])
plt.xlim(left=0)
plt.show()

**Interpretace grafu:**

Rozdělení cen je pravostranně zešikmené: většina nemovitostí se nachází v nižší až střední části cenového rozsahu a jen menší část dosahuje velmi vysokých cen. Dlouhý pravý chvost znamená, že několik drahých objektů může výrazně zvýšit průměrnou cenu. Proto je při popisu typické nemovitosti vhodné sledovat také medián, který je vůči extrémně vysokým cenám méně citlivý.

### 5.2. Bivariační analýza: Vztah vzdálenosti od centra a ceny

Prověříme, zda souvisí vzdálenost od centrální obchodní oblasti (CBD) s cenou nemovitosti. Bodový graf ukáže směr vztahu i to, jak velké jsou rozdíly v cenách při podobné vzdálenosti.

In [ ]:
plt.figure(figsize=(12, 6))
sns.scatterplot(data=df_clean, x='Distance', y='Price', alpha=0.3, color='crimson')
plt.title('Vztah mezi vzdáleností od centra (Distance) a cenou', fontsize=16)
plt.xlabel('Vzdálenost od CBD (km)', fontsize=12)
plt.ylabel('Cena (osa je zobrazena v milionech AUD)', fontsize=12)

yticks = plt.yticks()[0]
plt.yticks(ticks=yticks, labels=[f'{y/1e6:.1f}M' for y in yticks])
plt.ylim(bottom=0)
plt.show()

**Interpretace grafu:**

Graf naznačuje negativní vztah: dražší nemovitosti se častěji vyskytují blíže centru. Zároveň je patrný velký rozptyl cen při podobné vzdálenosti, takže samotná vzdálenost cenu nevysvětluje. Ve větší vzdálenosti se stále mohou nacházet drahé objekty a v blízkosti centra také levnější nemovitosti. Z grafu proto nelze tvrdit, že cena klesá přesně exponenciálně; jde pouze o vizuálně pozorovaný trend, který ovlivňují i další vlastnosti a lokalita.

### 5.3. Geografická distribuce: Cena podle regionu (`Regionname`)

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_clean, y='Regionname', x='Price', hue='Regionname', palette='viridis', legend=False)
plt.title('Rozdělení cen podle regionů Melbourne', fontsize=16)
plt.xlabel('Cena (osa je zobrazena v milionech AUD)', fontsize=12)
plt.ylabel('Region', fontsize=12)

xticks = plt.xticks()[0]
plt.xticks(ticks=xticks, labels=[f'{x/1e6:.1f}M' for x in xticks])
plt.xlim(left=0)
plt.show()

**Interpretace grafu:**

Boxplot umožňuje porovnat medián ceny, rozptyl prostředních 50 % pozorování a výskyt vzdálených hodnot v jednotlivých regionech. Regiony se liší jak typickou cenou, tak rozptylem cen. Vyšší počet bodů za hranicí vousů ukazuje, že v regionu existují i nemovitosti výrazně dražší než většina ostatních. Samotný boxplot ale neukazuje, zda rozdíly způsobuje typ nemovitosti, její plocha, stáří nebo jiné vlastnosti; to bychom museli dále ověřit.

### 5.4. Univariační analýza: Zastoupení typů nemovitostí (`Type`)

Po porovnání regionů se podíváme také na to, jak často se v datasetu objevují jednotlivé typy nemovitostí. Kódy znamenají `h` – dům, `u` – byt/jednotka a `t` – townhouse. Nerovnoměrné zastoupení kategorií je důležité při interpretaci klastrů: skupina s malým počtem pozorování může být méně stabilní a její mediány je vhodné posuzovat opatrně.

In [ ]:
# Počet pozorování v jednotlivých kategoriích Type
order_type = df_clean['Type'].value_counts().index
sns.countplot(data=df_clean, x='Type', order=order_type, color='steelblue')
plt.title('Počet nemovitostí podle typu', fontsize=16)
plt.xlabel('Typ nemovitosti')
plt.ylabel('Počet nemovitostí')

labels_type_plot = {
    'h': 'h – dům',
    'u': 'u – byt/jednotka',
    't': 't – townhouse'
}
plt.xticks(
    ticks=range(len(order_type)),
    labels=[labels_type_plot.get(t, t) for t in order_type]
)
plt.show()

**Interpretace grafu:**

Graf ukazuje, zda jsou jednotlivé typy zastoupeny rovnoměrně. V datasetu převažují domy (`h`), zatímco townhouse (`t`) jsou méně časté. To znamená, že celkové výsledky EDA a shlukování budou do určité míry odrážet hlavně vlastnosti častěji zastoupených domů. Proto je užitečné kontrolovat i podíl jednotlivých typů uvnitř každého klastru, nikoli pouze celkové počty v datasetu.

### 5.5. Multivariační analýza: Korelační matice

Korelační matice shrnuje směr a sílu lineárních vztahů mezi číselnými proměnnými. Pomáhá odhalit podobné příznaky, ale neprokazuje příčinnou souvislost.

In [ ]:
numerical_cols = df_clean.select_dtypes(include=[np.number]).drop(columns=['Postcode', 'Propertycount', 'Lattitude', 'Longtitude'])
corr_matrix = numerical_cols.corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title('Korelační matice klíčových metrik', fontsize=16)
plt.show()

**Interpretace a rozhodnutí pro další analýzu:**

1. Matice ukazuje silnou kladnou korelaci mezi `Rooms` a `Bedroom2`. Tyto proměnné popisují podobnou vlastnost, proto do clusterování zařadíme pouze `Rooms`, aby se stejná informace zbytečně nepočítala dvakrát.
2. `Price` má kladnou souvislost s obytnou plochou (`BuildingArea`) a počtem pokojů (`Rooms`). To odpovídá očekávání, že větší nemovitosti mohou být dražší, ale korelace sama o sobě nedokazuje příčinnou souvislost.
3. `Price` a `PricePerArea` do clusterování nezařadíme. Chceme skupiny tvořit podle fyzických a prostorových vlastností nemovitostí, nikoli přímo podle jejich tržní ceny. Cenu později použijeme až k interpretaci výsledných skupin.

Je také třeba počítat s tím, že `BuildingAge` je odvozený z `YearBuilt` a data prodeje, takže jeho silná souvislost s rokem výstavby je očekávaným důsledkem konstrukce příznaku.

## 6. Cluster Analysis (Shluková analýza)

Použijeme dva algoritmy: **K-Means** a **Agglomerative Hierarchical Clustering**. Nejprve připravíme vstupní proměnné, poté zdůvodníme počet skupin pomocí metrik a nakonec popíšeme jednotlivé klastry z hlediska trhu s nemovitostmi.

### 6.1. Příprava dat (Feature Selection & Scaling)

Před shlukováním upravíme vstupní data:

1. **Vyloučení ceny:** `Price` a `PricePerArea` nebudou vstupovat do clusterování. Použijeme je až při interpretaci skupin, abychom se vyhnuli přímému rozdělení podle cílové tržní hodnoty.
2. **Redundantní proměnná:** `Bedroom2` vynecháme, protože obsahuje informaci podobnou proměnné `Rooms`.
3. **Kategorická proměnná:** `Type` převedeme pomocí one-hot encodingu na číselné sloupce.
4. **Škálování:** Pomocí `StandardScaler` převedeme proměnné na srovnatelnou škálu, což je důležité pro algoritmy založené na vzdálenosti.
5. **Omezení odvozeného příznaku:** `BathroomPerRoom` je vypočten z `Bathroom` a `Rooms`. Jeho zahrnutí dává poměru koupelen a pokojů samostatný význam, ale zároveň těmto informacím přidává další váhu. Tuto skutečnost zohledníme při interpretaci výsledků.

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. Výběr příznaků pro clustering (bez Price, PricePerArea a duplicitního Bedroom2)
features_to_cluster = ['Rooms', 'Bathroom', 'BuildingArea', 'BuildingAge', 
                       'Distance', 'Landsize', 'BathroomPerRoom', 'Type']

df_cluster = df_clean[features_to_cluster].copy()

# 2. One-Hot Encoding pro kategorickou proměnnou 'Type'
df_cluster = pd.get_dummies(df_cluster, columns=['Type'], drop_first=False)

# 3. Standardizace dat
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_cluster)

# Vytvoření DataFrame s původními názvy sloupců pro lepší orientaci
df_scaled = pd.DataFrame(scaled_data, columns=df_cluster.columns)

print("=== DATA PŘIPRAVENA PRO CLUSTERING ===")
display(df_scaled.head())

### 6.2. Hledání vhodného počtu klastrů (Elbow Method & Silhouette Score)

Počet klastrů `k` zvolíme s pomocí dvou metrik:

1. **Inertia (metoda lokte):** Součet čtverců vzdáleností pozorování od středu příslušného klastru. S rostoucím počtem klastrů klesá; hledáme oblast, kde se další pokles začne zpomalovat.
2. **Silhouette Score:** Porovnává, jak blízko je pozorování vlastnímu klastru oproti nejbližšímu jinému klastru. Hodnota se pohybuje od −1 do 1; vyšší hodnota obvykle znamená lépe oddělené skupiny.

Otestujeme hodnoty `k` od 2 do 10. Elbow křivka nemusí vždy ukázat jeden naprosto jednoznačný bod, proto ji budeme posuzovat společně se Silhouette Score, nikoli jako samostatný důkaz.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

inertia = []
silhouette_scores = []
k_range = range(2, 11)

# Otestujeme několik hodnot počtu klastrů
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(df_scaled)
    inertia.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(df_scaled, kmeans.labels_))

# Přehled číselných hodnot umožňuje ověřit závěr i bez odečítání grafu
metric_table = pd.DataFrame({
    'k': list(k_range),
    'Inertia': inertia,
    'Silhouette Score': silhouette_scores
})
display(metric_table.round(3))
best_k_silhouette = int(
    metric_table.loc[metric_table['Silhouette Score'].idxmax(), 'k']
)
print(f"Nejvyšší Silhouette Score vychází pro k = {best_k_silhouette}.")
if best_k_silhouette != 3:
    print("UPOZORNĚNÍ: Maximum Silhouette Score není při k = 3. Před interpretací znovu zvažte počet klastrů a odpovídající popisy.")

# Vizualizace obou metrik
fig, ax1 = plt.subplots(figsize=(12, 6))
color_inertia = 'tab:blue'
ax1.set_xlabel('Počet klastrů (k)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Inertia (metoda lokte)', color=color_inertia, fontsize=12)
ax1.plot(k_range, inertia, marker='o', color=color_inertia, linewidth=2)
ax1.tick_params(axis='y', labelcolor=color_inertia)

ax2 = ax1.twinx()
color_silhouette = 'tab:red'
ax2.set_ylabel('Silhouette Score', color=color_silhouette, fontsize=12)
ax2.plot(k_range, silhouette_scores, marker='s', color=color_silhouette, linewidth=2)
ax2.tick_params(axis='y', labelcolor=color_silhouette)

plt.title('Výběr počtu klastrů: Inertia a Silhouette Score', fontsize=16)
fig.tight_layout()
plt.show()

**Matematické zdůvodnění:**

Silhouette Score dosahuje nejvyšší hodnoty přibližně **0,40 při `k = 3`**. Křivka Inertia s rostoucím `k` stále klesá, ale její pokles se postupně zpomaluje; „loket“ není zcela ostrý, a proto jej bereme jako podpůrný ukazatel. Volbu tří klastrů opíráme především o Silhouette Score a následně ji porovnáme s dendrogramem hierarchického shlukování.

Po změně postupu čištění je potřeba zkontrolovat tabulku metrik po opětovném spuštění. Kód vypíše, pro jaké `k` je Silhouette Score nejvyšší; pokud to nebude `k = 3`, je nutné zvážit volbu počtu klastrů znovu.

### 6.3. Vyhodnocení metrik a finální model K-Means

Pro interpretaci a porovnání s hierarchickým shlukováním použijeme **tři klastry (`k = 3`)**. Silhouette Score hodnotí oddělenost skupin, zatímco Inertia ukazuje, jak se mění vnitroskupinová variabilita při přidávání dalších klastrů. Tyto metriky poskytují společné, ale ne absolutní zdůvodnění volby. Před konečným závěrem zkontrolujeme, zda tabulka metrik po spuštění stále ukazuje maximum Silhouette Score při `k = 3`.

In [ ]:
# 1. Finální model K-Means se třemi klastry
kmeans_final = KMeans(n_clusters=3, random_state=42, n_init=10)
raw_cluster_labels = kmeans_final.fit_predict(df_scaled)

# Čísla klastrů jsou libovolná. Pro přehlednější interpretaci je po trénování
# seřadíme podle mediánové ceny (od nejnižší po nejvyšší). Cena není vstupem modelu.
raw_cluster_medians = (
    df_clean.assign(_raw_cluster=raw_cluster_labels)
    .groupby('_raw_cluster')['Price']
    .median()
    .sort_values()
)
cluster_label_map = {
    raw_label: new_label
    for new_label, raw_label in enumerate(raw_cluster_medians.index)
}
df_clean['Cluster_KMeans'] = pd.Series(raw_cluster_labels).map(cluster_label_map).to_numpy()
print('Štítky klastrů jsou pro přehlednost seřazeny podle mediánu ceny: 0 = nejnižší, 2 = nejvyšší.')

# 2. Profily klastrů v původních jednotkách
cluster_summary = df_clean.groupby('Cluster_KMeans').agg(
    Count=('Price', 'size'),
    MedianPrice=('Price', 'median'),
    MedianBuildingArea=('BuildingArea', 'median'),
    MedianLandsize=('Landsize', 'median'),
    MedianDistance=('Distance', 'median'),
    MedianBuildingAge=('BuildingAge', 'median'),
    MedianRooms=('Rooms', 'median'),
    MedianBathrooms=('Bathroom', 'median')
)

# Cena je přehledně zobrazena v milionech AUD; interní souhrn zůstává číselný
cluster_summary_display = cluster_summary.copy()
cluster_summary_display['MedianPrice (mil. AUD)'] = (
    cluster_summary_display['MedianPrice'] / 1e6
).round(2)
cluster_summary_display = cluster_summary_display.drop(columns='MedianPrice')

print('=== PROFILY KLASTRŮ (MEDIÁNY A POČET POZOROVÁNÍ) ===')
display(cluster_summary_display.round(2))

# 3. Podíl typů nemovitostí uvnitř každého klastru
labels_type = {
    'h': 'h – dům',
    'u': 'u – byt/jednotka',
    't': 't – townhouse'
}
type_share = pd.crosstab(
    df_clean['Cluster_KMeans'],
    df_clean['Type'],
    normalize='index'
).mul(100).round(1)
type_share = type_share.rename(columns=labels_type)

print('=== ZASTOUPENÍ TYPŮ NEMOVITOSTÍ V KLASTRECH (%) ===')
display(type_share)

### 6.4. Interpretace klastrů (Real-world Domain Logic)

Aby bylo možné klastrům přiřadit srozumitelné názvy, kód po dokončení K-Means seřadí jejich číselné štítky podle mediánu ceny: **klastr 0 má nejnižší medián ceny, klastr 1 prostřední a klastr 2 nejvyšší**. Cena nebyla použita při trénování; slouží pouze k přehlednějšímu označení již vytvořených skupin. Konkrétní profily jsou uvedeny v tabulce mediánů a zastoupení typů v navazující tabulce.

1. **Klastr 0 – cenově dostupnější a menší nemovitosti:** Tabulka mediánů ukazuje, že tato skupina má nejnižší medián ceny. Zároveň se vyznačuje menší obytnou plochou, nižším počtem pokojů a relativně malou mediánovou vzdáleností od CBD. To odpovídá segmentu menších a dostupnějších nemovitostí. Zastoupení bytů, domů a townhouse je uvedeno v tabulce podílů; pouze na základě mediánů nelze tvrdit, že jde převážně o byty.

2. **Klastr 1 – střední cenový segment s mladšími a prostornějšími objekty:** Tato skupina má podle profilu nižší medián stáří a velkou mediánovou obytnou plochu. To může odpovídat novějším a prostornějším nemovitostem. Přesnou charakteristiku je třeba odvozovat z aktuálních hodnot ve výstupní tabulce.

3. **Klastr 2 – nejdražší segment se starší zástavbou:** Tato skupina má nejvyšší medián ceny a vyšší medián stáří budovy. Jde tedy o dražší segment se staršími nemovitostmi. Samotné výsledky však nedokazují, že vyšší cenu způsobuje historická hodnota, prestiž čtvrti nebo velikost pozemku; takové vysvětlení by vyžadovalo další analýzu.

Mediánová vzdálenost od CBD u středního a nejdražšího segmentu je podobná, proto nelze tvrdit, že se nejdražší skupina nachází výrazně blíže či dále od centra. Výsledky popisují podobnosti v datech, nikoli příčiny cenových rozdílů.

### 6.5. Hierarchické shlukování (Agglomerative Clustering & Dendrogram)

Jako druhý algoritmus použijeme aglomerativní hierarchické shlukování s **Wardovou metodou**, která při každém kroku vybírá sloučení s nejmenším přírůstkem vnitroskupinového rozptylu.

Výpočet hierarchického shlukování pro všechna pozorování je výpočetně a paměťově náročný. Proto vytvoříme dendrogram na náhodném vzorku **3 000 pozorování**. Tento graf ukáže strukturu shlukování na vzorku, nikoli na celém datasetu.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score

# Náhodný vzorek; df_scaled i df_clean mají po přípravě stejný pořádek řádků
sample_idx = df_scaled.sample(n=3000, random_state=42).index
X_sample = df_scaled.loc[sample_idx]

# Wardovo hierarchické shlukování
ward_linkage = linkage(X_sample, method='ward')

# Výška řezu se vypočítá z posledních slučovacích vzdáleností tak,
# aby řez na celém stromu odděloval tři skupiny. Nemusíme tedy odhadovat pevnou hodnotu.
cut_height = (ward_linkage[-3, 2] + ward_linkage[-2, 2]) / 2

plt.figure(figsize=(14, 6))
dendrogram(ward_linkage, truncate_mode='lastp', p=20, show_leaf_counts=True)
plt.axhline(
    y=cut_height,
    color='red',
    linestyle='--',
    label=f'Řez pro 3 klastry (výška ≈ {cut_height:.1f})'
)
plt.title('Zkrácený dendrogram – hierarchické shlukování (Ward)', fontsize=16)
plt.xlabel('Sloučené skupiny (zkrácený dendrogram)', fontsize=12)
plt.ylabel('Vzdálenost sloučení (Ward)', fontsize=12)
plt.legend()
plt.show()
print(f"Výška řezu vypočítaná pro tři klastry: {cut_height:.3f}")

**Matematické zdůvodnění:**

Dendrogram znázorňuje, v jakých vzdálenostech se jednotlivé skupiny postupně spojují. Velký skok ve výšce sloučení může naznačovat, že se spojují již poměrně odlišné skupiny. Červená čára se v kódu neumisťuje na pevnou, předem odhadnutou hodnotu: její výška se vypočítá mezi dvěma posledními slučovacími vzdálenostmi tak, aby řez celým hierarchickým stromem vytvořil tři skupiny.

Jde o doplňující pohled na náhodný vzorek 3 000 pozorování. Proto jej nepovažujeme za důkaz, že jsou tři klastry ve všech ohledech nejlepší; výsledek porovnáme s K-Means pomocí metriky **Adjusted Rand Index (ARI)**.

In [ ]:
# Aglomerativní model se stejným počtem klastrů na stejném vzorku
agg_model = AgglomerativeClustering(n_clusters=3, linkage='ward')
sample_agg_labels = agg_model.fit_predict(X_sample)

# Štítky K-Means pro totožná pozorování
sample_kmeans_labels = df_clean['Cluster_KMeans'].iloc[sample_idx].to_numpy()

# ARI porovnává rozdělení bez ohledu na to, jakými čísly jsou klastry označeny
ari_score = adjusted_rand_score(sample_kmeans_labels, sample_agg_labels)
print(f"Adjusted Rand Index (ARI) mezi K-Means a hierarchickým shlukováním: {ari_score:.3f}")

**Interpretace ARI:**

Adjusted Rand Index porovnává, do jaké míry dva algoritmy přiřadily stejná pozorování do stejných dvojic skupin. Hodnota blízká **1** znamená velmi podobné rozdělení, hodnota kolem **0** shodu srovnatelnou s náhodným rozdělením a záporná hodnota horší shodu než náhodné očekávání. Konkrétní hodnota je uvedena ve výstupu předchozí buňky.

Vysoká hodnota ARI by znamenala, že se oba algoritmy na zvoleném vzorku ve velké míře shodují. Neznamená to však automaticky, že jsou klastry objektivně „správné“: oba modely pracují se stejnými vstupními příznaky a podobnost může ovlivnit také jejich škálování a výběr proměnných.

---

## 7. Závěr 1. části projektu (Method-Selection Memo)

V první části projektu jsme provedli průzkumnou analýzu a připravili data pro shlukování.

1. **Čištění dat:** Chybějící obytnou plochu a rok výstavby jsme doplnili skupinovými mediány se záložními možnostmi. Neznámé hodnoty `CouncilArea` zachováváme jako kategorii `Unknown`, abychom zbytečně neztratili pozorování. Zjevně neplatné nebo pro zvolený rozsah neobvyklé hodnoty jsme ošetřili doménovými filtry.
2. **Tvorba příznaků:** Vytvořili jsme `BuildingAge`, `PricePerArea` a `BathroomPerRoom`. Tyto příznaky popisují stáří budovy, cenu za metr čtvereční a poměr koupelen k počtu pokojů.
3. **EDA:** Grafy ukázaly rozdělení cen, vztah ceny ke vzdálenosti od CBD, rozdíly mezi regiony a korelace mezi číselnými proměnnými. Každý graf jsme doplnili interpretací a upozornili na omezení toho, co z něj lze vyvozovat.
4. **Shlukování:** Použili jsme K-Means a aglomerativní hierarchické shlukování s Wardovou metodou. Počet tří klastrů jsme zdůvodnili Silhouette Score, průběhem Inertia a kontrolou dendrogramu; shodu rozdělení porovnáváme pomocí ARI.
5. **Interpretace:** Výsledné skupiny popisujeme podle mediánu ceny, plochy, velikosti pozemku, vzdálenosti, stáří, počtu pokojů a zastoupení typů nemovitostí. Tyto charakteristiky ukazují rozdíly mezi segmenty, ale samy o sobě neprokazují příčiny cenových rozdílů.

Pro reprodukovatelnost používají oba algoritmy pevně nastavený `random_state`; výsledné tabulky a grafy lze po spuštění notebooku znovu vytvořit ze zdrojového CSV.